In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [7]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [9]:
!pip install -U "autogluon.tabular[tabarena]" --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.4/248.4 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.7/90.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.4/334.4 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.9/252.9 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 732.3/732.3 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 111.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.8/514.8 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [16]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
if(torch.cuda.is_available):
    print(torch.cuda.get_device_name(0))

2.11.0+cu128
True
Tesla T4


In [17]:
# ============================================================
# PHASE 10A — DATA PREPARATION
# ============================================================

import pandas as pd
import numpy as np

 # "/kaggle/input/playground-series-s6e10/train.csv"
# "/kaggle/input/playground-series-s6e10/test.csv"
TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e10/train.csv"
TEST_PATH  = "/kaggle/input/competitions/playground-series-s6e10/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("=" * 70)
print("DATASET")
print("=" * 70)

print("Train:", train.shape)
print("Test :", test.shape)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nTarget:")
print(train["satisfaction"].value_counts())
print(train["satisfaction"].value_counts(normalize=True))

print("\nMissing values:")
print(train.isna().sum()[train.isna().sum() > 0])

print("\nDtypes:")
print(train.dtypes)

DATASET
Train: (699635, 23)
Test : (299844, 22)

Train columns:
['id', 'Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Gender', 'Customer Type', 'Type of Travel', 'Class', 'satisfaction']

Target:
satisfaction
False    389296
True     310339
Name: count, dtype: int64
satisfaction
False    0.556427
True     0.443573
Name: proportion, dtype: float64

Missing values:
Arrival Delay in Minutes    292
dtype: int64

Dtypes:
id                                     int64
Age                                    int64
Flight Distance                        int64
Inflight wifi service                  int64
Departure/Arrival time convenient      int64
Ease of Online booking                 

In [18]:
# ============================================================
# PHASE 10B — AUTOGluon DATA PREPARATION
# ============================================================

from autogluon.tabular import TabularPredictor

TARGET = "satisfaction"

# Copy data
ag_train = train.copy()
ag_test = test.copy()

# Drop ID
ag_train = ag_train.drop(columns=["id"])
ag_test = ag_test.drop(columns=["id"])

# Convert target to integer
# AutoGluon can handle bool, but integer makes the binary target explicit.
ag_train[TARGET] = ag_train[TARGET].astype(int)

# Explicitly mark categorical columns
categorical_cols = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
]

for col in categorical_cols:
    ag_train[col] = ag_train[col].astype("category")
    ag_test[col] = ag_test[col].astype("category")

print("=" * 70)
print("AUTOGLOON DATA")
print("=" * 70)

print("Train:", ag_train.shape)
print("Test :", ag_test.shape)

print("\nTarget:")
print(ag_train[TARGET].value_counts())
print(ag_train[TARGET].value_counts(normalize=True))

print("\nCategorical columns:")
for col in categorical_cols:
    print(f"{col:<40} {ag_train[col].dtype}")

print("\nMissing values:")
print(ag_train.isna().sum()[ag_train.isna().sum() > 0])

AUTOGLOON DATA
Train: (699635, 22)
Test : (299844, 21)

Target:
satisfaction
0    389296
1    310339
Name: count, dtype: int64
satisfaction
0    0.556427
1    0.443573
Name: proportion, dtype: float64

Categorical columns:
Gender                                   category
Customer Type                            category
Type of Travel                           category
Class                                    category

Missing values:
Arrival Delay in Minutes    292
dtype: int64


In [19]:
# ============================================================
# PHASE 10C — AUTOGLUON CONTROLLED BASE RUN
# ============================================================

from autogluon.tabular import TabularPredictor
import os

AG_PATH = "/kaggle/working/autogluon_phase10_base"

predictor = TabularPredictor(
    label="satisfaction",
    problem_type="binary",
    eval_metric="roc_auc",
    path=AG_PATH
)

predictor.fit(
    train_data=ag_train,
    presets="medium",
    time_limit=3600,
    num_bag_folds=5,
    num_stack_levels=0,
    verbosity=2
)

Preset alias specified: 'medium' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.13.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Sun Sep 27 17:32:16 UTC 2026
CPU Count:          4
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB | GPU 1: 14.56/14.56 GB
Total GPU Memory:   Free: 29.12 GB, Allocated: 0.00 GB, Total: 29.12 GB
GPU Count:          2
Memory Avail:       28.86 GB / 31.35 GB (92.1%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['medium']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "/kaggle/working/autogluon_phase10_base"
Train Data Rows:    699635
Train Data Columns: 21
Label Column:       satisfaction
Problem Type:       binary
Preprocessing data...
Selected class <-

Target shape: (699635,)
AG OOF shape: (699635,)
AG TEST shape: (299844,)

AUC:
AutoGluon OOF : 0.9728586079452893


NameError: name 'CHAMPION_OOF' is not defined

In [24]:
# ============================================================
# PHASE 10B — RECOVER TARGET + FINISH METRICS
# ============================================================

import numpy as np
from sklearn.metrics import roc_auc_score

# Recover target safely
if "y" not in globals():
    y = ag_train["satisfaction"].to_numpy()

print("Target shape:", y.shape)
print("AG OOF shape:", ag_oof.shape)
print("AG TEST shape:", ag_test.shape)

# ------------------------------------------------------------
# AUC
# ------------------------------------------------------------

print("\nAUC:")
print("AutoGluon OOF :", roc_auc_score(y, ag_oof))
print("Champion OOF  :", roc_auc_score(y, CHAMPION_OOF))

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print("\nPrediction statistics:")
print("AG mean        :", ag_oof.mean())
print("Champion mean  :", CHAMPION_OOF.mean())

print("AG min/max     :", ag_oof.min(), ag_oof.max())
print("Champion min/max:",
      CHAMPION_OOF.min(), CHAMPION_OOF.max())

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

np.save(
    "/kaggle/working/phase10_ag_raw_oof.npy",
    ag_oof
)

np.save(
    "/kaggle/working/phase10_ag_raw_test.npy",
    ag_test
)

print("\nSaved successfully.")

Target shape: (699635,)
AG OOF shape: (699635,)
AG TEST shape: (299844,)

AUC:
AutoGluon OOF : 0.9728586079452893


NameError: name 'CHAMPION_OOF' is not defined

In [22]:
# ============================================================
# PHASE 10B — EXTRACT AUTOGLOUON PREDICTIONS
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from autogluon.tabular import TabularPredictor

AG_PATH = "/kaggle/working/autogluon_phase10_base"

predictor = TabularPredictor.load(AG_PATH)

print("=" * 70)
print("PHASE 10B — AUTOGLUON PREDICTION EXTRACTION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Leaderboard
# ------------------------------------------------------------

ag_lb = predictor.leaderboard(silent=True)

display(ag_lb)

# Best model = first row of leaderboard
best_ag_model = ag_lb.iloc[0]["model"]

print("\nBest AutoGluon model:", best_ag_model)
print(
    "Leaderboard validation AUC:",
    ag_lb.iloc[0]["score_val"]
)

# ------------------------------------------------------------
# 2. OOF predictions
# ------------------------------------------------------------

ag_oof_df = predictor.predict_proba(
    ag_train,
    model=best_ag_model
)

print("\nOOF prediction columns:", ag_oof_df.columns.tolist())

# Positive-class probability
ag_oof = ag_oof_df[1].to_numpy(dtype=np.float64)

# ------------------------------------------------------------
# 3. Test predictions
# ------------------------------------------------------------

ag_test_df = predictor.predict_proba(
    ag_test,
    model=best_ag_model
)

ag_test = ag_test_df[1].to_numpy(dtype=np.float64)

# ------------------------------------------------------------
# 4. Checks
# ------------------------------------------------------------

print("\nShapes:")
print("AG OOF  :", ag_oof.shape)
print("AG TEST :", ag_test.shape)

print("\nAUC:")
print("AutoGluon OOF :", roc_auc_score(y, ag_oof))
print("Champion OOF  :", roc_auc_score(y, CHAMPION_OOF))

print("\nPrediction statistics:")
print("AG mean       :", ag_oof.mean())
print("Champion mean :", CHAMPION_OOF.mean())
print("AG min/max    :", ag_oof.min(), ag_oof.max())
print("Champion min/max:",
      CHAMPION_OOF.min(), CHAMPION_OOF.max())

# ------------------------------------------------------------
# 5. Save safely
# ------------------------------------------------------------

np.save(
    "/kaggle/working/phase10_ag_raw_oof.npy",
    ag_oof
)

np.save(
    "/kaggle/working/phase10_ag_raw_test.npy",
    ag_test
)

print("\nSaved:")
print("/kaggle/working/phase10_ag_raw_oof.npy")
print("/kaggle/working/phase10_ag_raw_test.npy")

PHASE 10B — AUTOGLUON PREDICTION EXTRACTION


,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,0.958953,roc_auc,144.938960,2824.048062,0.144993,35.016114,2,True,10
1,CatBoost_BAG_L1,0.958502,roc_auc,1.656681,2073.066678,1.656681,2073.066678,1,True,5
2,LightGBM_BAG_L1,0.958481,roc_auc,84.761425,229.778479,84.761425,229.778479,1,True,2
3,XGBoost_BAG_L1,0.957848,roc_auc,7.024250,77.850451,7.024250,77.850451,1,True,9
4,RandomForestGini_BAG_L1,0.957484,roc_auc,23.198520,135.493387,23.198520,135.493387,1,True,3
5,RandomForestEntr_BAG_L1,0.957381,roc_auc,27.202638,157.230863,27.202638,157.230863,1,True,4
6,NeuralNetFastAI_BAG_L1,0.957201,roc_auc,7.974703,193.462541,7.974703,193.462541,1,True,8
7,ExtraTreesEntr_BAG_L1,0.956374,roc_auc,24.368597,87.682484,24.368597,87.682484,1,True,7
8,ExtraTreesGini_BAG_L1,0.956308,roc_auc,22.140469,80.527980,22.140469,80.527980,1,True,6
9,LightGBMXT_BAG_L1,0.953560,roc_auc,135.064134,365.965708,135.064134,365.965708,1,True,1



Best AutoGluon model: WeightedEnsemble_L2
Leaderboard validation AUC: 0.9589526910241707

OOF prediction columns: [0, 1]

Shapes:
AG OOF  : (699635,)
AG TEST : (299844,)

AUC:


NameError: name 'y' is not defined

In [ ]:
# ============================================================
# PHASE 10B — EXTRACT AUTOGLOUON PREDICTIONS
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# Load predictor if needed
from autogluon.tabular import TabularPredictor

AG_PATH = "/kaggle/working/autogluon_phase10_base"

predictor = TabularPredictor.load(AG_PATH)

print("=" * 70)
print("PHASE 10B — AUTOGLUON PREDICTION EXTRACTION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Best AutoGluon model
# ------------------------------------------------------------

print("\nLeaderboard:")
ag_lb = predictor.leaderboard(silent=True)
display(ag_lb)

best_ag_model = predictor.get_model_best()
print("\nBest AutoGluon model:", best_ag_model)

# ------------------------------------------------------------
# 2. OOF predictions
# ------------------------------------------------------------

ag_oof = predictor.predict_proba(
    ag_train,
    model=best_ag_model
)

# Binary probability column
if isinstance(ag_oof, pd.DataFrame):
    ag_oof = ag_oof[1].to_numpy()
else:
    ag_oof = np.asarray(ag_oof)

# ------------------------------------------------------------
# 3. Test predictions
# ------------------------------------------------------------

ag_test = predictor.predict_proba(
    ag_test,
    model=best_ag_model
)

if isinstance(ag_test, pd.DataFrame):
    ag_test = ag_test[1].to_numpy()
else:
    ag_test = np.asarray(ag_test)

# ------------------------------------------------------------
# 4. Basic checks
# ------------------------------------------------------------

print("\nShapes:")
print("AG OOF  :", ag_oof.shape)
print("AG TEST :", ag_test.shape)

print("\nAUC:")
print("AutoGluon OOF :", roc_auc_score(y, ag_oof))
print("Champion OOF  :", roc_auc_score(y, CHAMPION_OOF))

print("\nPrediction statistics:")
print("AG mean       :", ag_oof.mean())
print("Champion mean :", CHAMPION_OOF.mean())
print("AG min/max    :", ag_oof.min(), ag_oof.max())

# ------------------------------------------------------------
# 5. Save safely
# ------------------------------------------------------------

np.save(
    "/kaggle/working/phase10_ag_raw_oof.npy",
    ag_oof
)

np.save(
    "/kaggle/working/phase10_ag_raw_test.npy",
    ag_test
)

print("\nSaved:")
print("/kaggle/working/phase10_ag_raw_oof.npy")
print("/kaggle/working/phase10_ag_raw_test.npy")

In [25]:
# ============================================================
# PHASE 10 — CHECKPOINT SAVE
# ============================================================

import os
import shutil

CHECKPOINT = "/kaggle/working/phase10_autogluon_checkpoint"
os.makedirs(CHECKPOINT, exist_ok=True)

# ------------------------------------------------------------
# 1. Copy AutoGluon predictor
# ------------------------------------------------------------

AG_SRC = "/kaggle/working/autogluon_phase10_base"
AG_DST = os.path.join(CHECKPOINT, "autogluon_phase10_base")

if os.path.exists(AG_DST):
    shutil.rmtree(AG_DST)

shutil.copytree(AG_SRC, AG_DST)

# ------------------------------------------------------------
# 2. Save predictions if they exist
# ------------------------------------------------------------

if "ag_oof" in globals():
    np.save(
        os.path.join(CHECKPOINT, "ag_raw_oof.npy"),
        ag_oof
    )
    print("✅ AG OOF saved")

if "ag_test" in globals():
    np.save(
        os.path.join(CHECKPOINT, "ag_raw_test.npy"),
        ag_test
    )
    print("✅ AG TEST saved")

# ------------------------------------------------------------
# 3. Save leaderboard
# ------------------------------------------------------------

if "ag_lb" in globals():
    ag_lb.to_csv(
        os.path.join(CHECKPOINT, "autogluon_leaderboard.csv"),
        index=False
    )
    print("✅ Leaderboard saved")

# ------------------------------------------------------------
# 4. Save experiment summary
# ------------------------------------------------------------

summary = """
PHASE 10 — AUTOGLUON RAW BASELINE
=================================

AutoGluon Version:
1.6.3

Environment:
Python 3.13.15
PyTorch 2.11.0 + CUDA 12.8
GPU: 2x Tesla T4

Dataset:
Train: 699635 rows
Test: 299844 rows
Features: 21
Target: satisfaction

Training:
Preset: medium / medium_quality
Time limit: 3600 seconds
num_bag_folds: 5
num_stack_levels: 0
Metric: ROC-AUC

Best model:
WeightedEnsemble_L2

Validation AUC:
0.9589526910241707

Models trained included:
LightGBMXT
LightGBM
RandomForestGini
RandomForestEntr
CatBoost
ExtraTreesGini
ExtraTreesEntr
NeuralNetFastAI
XGBoost
WeightedEnsemble_L2

IMPORTANT:
This AutoGluon model is NOT the current competition champion.

Current protected champion:
135-model stack
CV AUC approximately 0.96208
Kaggle LB approximately 0.96177
Rank approximately #11

NEXT:
Phase 10C — Compare AutoGluon predictions against
the 135-model champion for prediction/residual diversity.

DO NOT overwrite champion artifacts.
"""

with open(
    os.path.join(CHECKPOINT, "phase10_summary.txt"),
    "w"
) as f:
    f.write(summary)

print("✅ Summary saved")

# ------------------------------------------------------------
# 5. Show checkpoint contents
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CHECKPOINT CONTENTS")
print("=" * 70)

for root, dirs, files in os.walk(CHECKPOINT):
    level = root.replace(CHECKPOINT, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        size_mb = os.path.getsize(os.path.join(root, file)) / (1024**2)
        print(f"{indent}  {file} ({size_mb:.1f} MB)")

✅ AG OOF saved
✅ AG TEST saved
✅ Leaderboard saved
✅ Summary saved

CHECKPOINT CONTENTS
phase10_autogluon_checkpoint/
  autogluon_leaderboard.csv (0.0 MB)
  phase10_summary.txt (0.0 MB)
  ag_raw_test.npy (2.3 MB)
  ag_raw_oof.npy (5.3 MB)
  autogluon_phase10_base/
    metadata.json (0.0 MB)
    version.txt (0.0 MB)
    predictor.pkl (0.0 MB)
    learner.pkl (0.0 MB)
    utils/
      data/
        X.pkl (93.4 MB)
        y.pkl (0.7 MB)
    models/
      trainer.pkl (0.0 MB)
      LightGBM_BAG_L1/
        model.pkl (0.0 MB)
        S1F4/
          model.pkl (2.8 MB)
        S1F5/
          model.pkl (4.9 MB)
        utils/
          oof.pkl (6.0 MB)
          model_template.pkl (0.0 MB)
        S1F3/
          model.pkl (4.4 MB)
        S1F2/
          model.pkl (4.7 MB)
        S1F1/
          model.pkl (4.6 MB)
      ExtraTreesEntr_BAG_L1/
        model.pkl (0.0 MB)
        utils/
          oof.pkl (3.3 MB)
          model_template.pkl (0.0 MB)
        S1F1/
          model.pkl (686.7 